# Notebook for the LLM processing of medical reports

In [1]:
%load_ext autoreload
%autoreload 2

# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
import pandas as pd

The data contains text documents that are annotated for mentions of participants, interventions and outcomes (PIO) in medical research. For each entity type, P, I, or O, there is a slightly different set of documents in the training and test set. Most of the documents are identical, but each type has a few extra documents. Looking at the lecture notes and doing some research online it seems clear that the best approach currently available for the task of token classification are transformer models, specifically transformer encoders, the most famous example being the BERT (Bidirectional Encoder Representations from Transformers) model developed at Google. In order to test these models a group called Huggingfaces has created a website where people can upload their deep learning models and they have many examples of different BERT models that have been developed as well as providing a tutorial on how to use the API they have developed for quickly loading and utilising these models. The model I have looked at is one of the smallest versions: DistilBERT

However, first we need to load the data from the files so here I used the code provided.    
To load the text documents, we first make a list of the document IDs for one entity type (P, I or O):

In [2]:
from pathlib import Path

DATA_DIR = Path("./ebm_nlp_2_00")

docs_dir = DATA_DIR / "documents"

def get_doc_ids(split="train", label_type="interventions"):
    """ 
    split: 'train' or 'test' 
    """

    if split == "test":
        split = "test/gold"

    train_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type  # assuming that the split is the same for all entity types, we can just look at one of them
        / split
    )
    
    doc_ids = [p.stem.split(".")[0] for p in train_dir.glob("*.AGGREGATED.ann")]
   # print(doc_ids)
    return sorted(doc_ids)

doc_ids_i = get_doc_ids("train", "interventions")
test_doc_ids_i = get_doc_ids("test", "interventions")

print(f"Number of documents in train split for interventions: {len(doc_ids_i)}")
print(f"Number of documents in test split for interventions: {len(test_doc_ids_i)}")

Number of documents in train split for interventions: 4746
Number of documents in test split for interventions: 187


In [3]:
def load_labels_for_doc(doc_id, label_type="interventions", split="train"):
    """
    label_type: 'participants', 'interventions', or 'outcomes'
    split: 'train' or 'test' 
    """
    if split == "test":
        split = "test/gold"

    ann_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / label_type / split/ f"{doc_id}.AGGREGATED.ann"
    
    if not ann_path.exists():
        print(ann_path, "does not exist!")
        return None
    
    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f]
    
    return labels

def load_labels(doc_ids, label_type="interventions", split="train"):
    labels = []
    for doc_id in doc_ids:
        doc_labels = load_labels_for_doc(doc_id, label_type, split)
        if doc_labels is not None:
            labels.append(doc_labels)
    return labels

interventions_labels = load_labels(doc_ids_i, "interventions", split="train")

print(f"Length of participants_labels: {len(interventions_labels)}")

test_interventions_labels = load_labels(test_doc_ids_i, "interventions", split="test")
print(f"Length of test_participants_labels: {len(test_interventions_labels)}")

sample = 123
print("Document ID:", doc_ids_i[sample])
print(f"Interventions label example for doc {doc_ids_i[sample]}:")
print(interventions_labels[sample])

Length of participants_labels: 4746
Length of test_participants_labels: 187
Document ID: 10674680
Interventions label example for doc 10674680:
['0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', 

In [4]:
def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def load_documents(doc_ids):
    documents = []
    for doc_id in doc_ids:
        doc = load_document(doc_id)
        documents.append(doc)
    return documents

interventions_tokens = load_documents(doc_ids_i)
test_interventions_tokens = load_documents(test_doc_ids_i)

# inspect a random element
print("Document ID:", doc_ids_i[sample])
print(f"Tokenised document example for doc {doc_ids_i[sample]}:")
print(interventions_tokens[sample])
print(interventions_labels[sample])

Document ID: 10674680
Tokenised document example for doc 10674680:
['Assessment', 'of', 'therapeutic', 'response', 'of', 'Plasmodium', 'falciparum', 'to', 'chloroquine', 'and', 'sulfadoxine-pyrimethamine', 'in', 'an', 'area', 'of', 'low', 'malaria', 'transmission', 'in', 'Colombia', '.', 'Although', 'chloroquine', '(', 'CQ', ')', 'resistance', 'was', 'first', 'reported', 'in', 'Colombia', 'in', '1961', 'and', 'sulfadoxine-pyrimethamine', '(', 'SP', ')', 'resistance', 'in', '1981', ',', 'the', 'frequency', 'of', 'treatment', 'failures', 'to', 'these', 'drugs', 'in', 'Colombia', 'is', 'unclear', '.', 'A', 'modified', 'World', 'Health', 'Organization', '14-day', 'in', 'vivo', 'drug', 'efficacy', 'test', 'for', 'uncomplicated', 'Plasmodium', 'falciparum', 'malaria', 'in', 'areas', 'with', 'intense', 'malaria', 'transmission', 'was', 'adapted', 'to', 'reflect', 'the', 'clinical', 'and', 'epidemiologic', 'features', 'of', 'a', 'low-intensity', 'malaria', 'transmission', 'area', 'in', 'the', 

In [5]:
def load_text_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.txt"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

text = load_text_document(10674680)

text

['Assessment of therapeutic response of Plasmodium falciparum to chloroquine and sulfadoxine-pyrimethamine in an area of low malaria transmission in Colombia.',
 '',
 'Although chloroquine (CQ) resistance was first reported in Colombia in 1961 and sulfadoxine-pyrimethamine (SP) resistance in 1981, the frequency of treatment failures to these drugs in Colombia is unclear. A modified World Health Organization 14-day in vivo drug efficacy test for uncomplicated Plasmodium falciparum malaria in areas with intense malaria transmission was adapted to reflect the clinical and epidemiologic features of a low-intensity malaria transmission area in the Pacific Coast Region of Colombia. Patients > or =1 year of age with a parasite density > or =1,000 asexual parasites per microliter were enrolled in this study. Forty-four percent (24 of 54) of the CQ-treated patients were therapeutic failures, including 7 early treatment failures (ETFs) and 17 late treatment failures (LTFs). Four (6%) of 67 SP-tr

I decided to start the processing with identifying interventions. This is because I thought that this would be the simplest one as most of the interventions are named drugs so while each drug might be unique, the contexts in which they appear should all be similar.

### Process to follow:
1. Load the files
2. Convert labels to ones combining the B I O labels with the ner tags
3. Convert combined labels to numeric ids
4. Initialize the model and tokenizer
5. Tokenize the words and align the labels with any words that have been split into multiple tokens
6. Prepare the model
7. Create the compute metrics function
8. Add the training arguments
9. Run the model
10. Repeat for Participants and Outcomes

In [6]:
from itertools import chain
import numpy as np

all_labels = chain(*interventions_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['0' '1' '2' '3' '4' '5' '6' '7']


### Reformatting the labels

In order to do token classification it helps to label the starting points of tokens (typically noted as B) as well as points that are "inside" the token (i.e. the second part of a name) as this helps split tokens that are next to each other. As our data is labelled 0-7 as seen above we can alter the labels by adding "B-" or "I-" to the start of each one (except for the zeros which are replaced with the letter O). Therefore I modified the function we were given to convert the labels to teh ones required by the BERT model

In [7]:
def hierarchical_to_ner(tags):
    """
    Combine the EBM-NLP hierarchical labels (0–7) with BIO tags.

    Parameters
    ----------
    tags : list[int]
        A list of hierarchical labels for a single document.

    Returns
    -------
    list[str]
        BIO tags ("O", "B-1", "I-1", "B-2", "I-2"...).
    """

    bio = []
    prev = 0

    for t in tags:
        t = int(t)  # ensure it's an integer
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B-" + str(t))
            else:
                bio.append("I-" + str(t))
        prev = t
        

    return bio

def convert_all_labels_to_ner(labels):
    for i, doc_labels in enumerate(labels):
        labels[i] = hierarchical_to_ner(doc_labels)
    return labels

In [8]:
interventions_ner_labels = convert_all_labels_to_ner(interventions_labels)
test_interventions_ner_labels = convert_all_labels_to_ner(test_interventions_labels)

all_labels_updated = chain(*interventions_ner_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels_updated)))

['B-1' 'B-2' 'B-3' 'B-4' 'B-5' 'B-6' 'B-7' 'I-1' 'I-2' 'I-3' 'I-4' 'I-5'
 'I-6' 'I-7' 'O']


When using the huggingface API we need to create to dicts, to allow the model to convert the labels in to numbers and vice versa. So next we define the mappings from id to label and label to id

In [9]:
id2label_i = {
    0: "O",
    1: "B-1",
    2: "I-1",
    3: "B-2",
    4: "I-2",
    5: "B-3",
    6: "I-3",
    7: "B-4",
    8: "I-4",
    9: "B-5",
    10: "I-5",
    11: "B-6",
    12: "I-6",
    13: "B-7",
    14: "I-7",
}

label2id_i = {
    "O": 0,
    "B-1": 1,
    "I-1": 2,
    "B-2": 3,
    "I-2": 4,
    "B-3": 5,
    "I-3": 6,
    "B-4": 7,
    "I-4": 8,
    "B-5": 9,
    "I-5": 10,
    "B-6": 11,
    "I-6": 12,
    "B-7": 13,
    "I-7": 14,
}

Then we convert the ner tags to their respective ids

In [10]:
def label_to_ids(labels_list, labels2ids):
    ids_list = []
    for label_list in labels_list:
        ids = []
        for label in label_list:
            ids.append(labels2ids[label])
        ids_list.append(ids)
    return ids_list

In [11]:
interventions_ner_ids = label_to_ids(interventions_ner_labels, label2id_i)
test_interventions_ner_ids = label_to_ids(test_interventions_ner_labels, label2id_i)
print(interventions_ner_ids[0])
print(test_interventions_ner_ids[0])

[0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 5, 6, 6, 6, 0, 5, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 5, 6, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

Now since we are using a pretrained model we can use the pretrained tokenizer to tokenize our input words. However we need to be careful when doing this as some words may be broken up into several tokens resulting in the tokens and labels being misaligned. To solve this issue we can use the Huggingface inbuilt tokenizer field `batch_id` which keeps track of multiple tokens belonging to the same word. Here we give each additional part of each token the number "-100". This we be used later to allow the model to ignore these extra parts (Note: after testing the model while the accuracy was high, over 95%, the other metrics (precision, recall and f1-score) were all around 40% signalling that the data is heavily skewed towards negative results and the model is over predicting negative outputs. After changing this function to include the extra parts of each split token the 3 metrics rose to over 55% for interventions) 

In [12]:
def tokenize_and_align_labels(inputs):
    tokenized_inputs = tokenizer(inputs["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(inputs["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i) # This gives the same word id for words that have been split up
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx: # Only label the first token of a given word
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
                #label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

We then need to convert the data into an objects that can be used by the API: The huggingface Dataset and DataDict (The DataDict isn't technically necessary but it can help keep all the data together so it is very convenient. Also most of the datasets available on huggingfaces are in the format of a DataDict)

In [13]:
from datasets import Dataset, DatasetDict

interventions_initial_train_ds = Dataset.from_dict({
    "tokens": interventions_tokens,
    "ner_tags": interventions_ner_ids,
})

test_interventions_initial_train_ds = Dataset.from_dict({
    "tokens": test_interventions_tokens,
    "ner_tags": test_interventions_ner_ids,
})

In [14]:
ds_dict_i = DatasetDict({
    "train": interventions_initial_train_ds,
    "test": test_interventions_initial_train_ds,
})

Now to use the above function we first need to initialize our tokenizer.  
Our first test will be the smallest BERT model: [DistilBERT (base-uncased)](https://huggingface.co/distilbert/distilbert-base-uncased)

In [15]:
from transformers import AutoTokenizer, DistilBertForTokenClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

Also using the Dataset datatype allows us to use its inbuilt map function which updates the data more efficiently

In [16]:
interventions_ds = ds_dict_i.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4746 [00:00<?, ? examples/s]

Map:   0%|          | 0/187 [00:00<?, ? examples/s]

Now that the data has been tokenized we still need to worry about alignment between data samples. As not all documents are the same length most documents will need to be either padded or truncated. Initially I wrote a function to carry this out but then I found that the API can carry this out using a DataCollator which was much more straightforward

In [17]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [18]:
import evaluate

seqeval = evaluate.load("seqeval")

In [19]:
import numpy as np

label_list_i = list(label2id_i.keys())

def compute_metrics_i(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_i[p] for (p, l) in zip(prediction, label) if l != -100] # Use this to ignore everything but the first token for words that got split during tokenization
        for prediction, label in zip(predictions, labels)                   # or ignore special tokens added by tokenizer
    ]

    true_labels = [
        [label_list_i[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [20]:
model_i = DistilBertForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels = 15, id2label=id2label_i, label2id=label2id_i)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
from transformers import TrainingArguments, Trainer

In [22]:
training_args_i = TrainingArguments(
    output_dir = "model/interventions_classification",
    learning_rate = 1e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
   # optim="apollo_adamw",
   # optim_target_modules=[r".*.attention.*", r".*.ffn.*"],
    num_train_epochs = 5,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

In [23]:
trainer_i = Trainer(
    model = model_i,
    args = training_args_i,
    train_dataset = interventions_ds["train"],
    eval_dataset = interventions_ds["test"],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics_i,
)

In [24]:
trainer_i.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.188839,0.490000,0.365672,0.418803,0.954844
2,0.303793,0.181606,0.405161,0.426052,0.415344,0.949265
3,0.303793,0.173197,0.410294,0.378562,0.393790,0.949427
4,0.190670,0.166037,0.428670,0.421981,0.425299,0.951994
5,0.190670,0.161254,0.443239,0.415875,0.429121,0.952843


/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1485, training_loss=0.22532327054726958, metrics={'train_runtime': 276.1842, 'train_samples_per_second': 85.921, 'train_steps_per_second': 5.377, 'total_flos': 3087597482858772.0, 'train_loss': 0.22532327054726958, 'epoch': 5.0})

Whilst the accuracy score is high, the low f1 score indicates that the model is correctly predicting the negative case which makes up most of the data, but not the positive cases that we want to predict. 

In [25]:
trainer_i.save_model("model/interventions_classification")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Participants
Now to repeat the process with participants

In [26]:
doc_ids_p = get_doc_ids("train", "participants")
test_doc_ids_p = get_doc_ids("test", "participants")

print(f"Number of documents in train split for participants: {len(doc_ids_p)}")
print(f"Number of documents in test split for participants: {len(test_doc_ids_p)}")

participants_tokens = load_documents(doc_ids_p)
test_participants_tokens = load_documents(test_doc_ids_p)

Number of documents in train split for participants: 4609
Number of documents in test split for participants: 189


In [27]:
participants_labels = load_labels(doc_ids_p, "participants", split="train")
print(f"Length of participants_labels: {len(participants_labels)}")

test_participants_labels = load_labels(test_doc_ids_p, "participants", split="test")
print(f"Length of test_participants_labels: {len(test_participants_labels)}")

participants_ner_labels = convert_all_labels_to_ner(participants_labels)
test_participants_ner_labels = convert_all_labels_to_ner(test_participants_labels)

Length of participants_labels: 4609
Length of test_participants_labels: 189


In [28]:
all_labels_p = chain(*participants_ner_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels_p)))

['B-1' 'B-2' 'B-3' 'B-4' 'I-1' 'I-2' 'I-3' 'I-4' 'O']


This gives us the labels for the participants

In [29]:
id2label_p = {
    0: "O",
    1: "B-1",
    2: "I-1",
    3: "B-2",
    4: "I-2",
    5: "B-3",
    6: "I-3",
    7: "B-4",
    8: "I-4",
}

label2id_p = {
    "O": 0,
    "B-1": 1,
    "I-1": 2,
    "B-2": 3,
    "I-2": 4,
    "B-3": 5,
    "I-3": 6,
    "B-4": 7,
    "I-4": 8,
}

In [30]:
participants_ner_ids = label_to_ids(participants_ner_labels, label2id_p)
test_participants_ner_ids = label_to_ids(test_participants_ner_labels, label2id_p)

In [31]:
participants_initial_train_ds = Dataset.from_dict({
    "tokens": participants_tokens,
    "ner_tags": participants_ner_ids,
})

test_participants_initial_train_ds = Dataset.from_dict({
    "tokens": test_participants_tokens,
    "ner_tags": test_participants_ner_ids,
})

In [32]:
ds_dict_p = DatasetDict({
    "train": participants_initial_train_ds,
    "test": test_participants_initial_train_ds,
})

In [33]:
participants_ds = ds_dict_p.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4609 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

In [34]:
label_list_p = list(label2id_p.keys())

def compute_metrics_p(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_p[p] for (p, l) in zip(prediction, label) if l != -100] # Use this to ignore everything but the first token for words that got split during tokenization
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list_p[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [35]:
model_p = DistilBertForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels = 9, id2label=id2label_p, label2id=label2id_p)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
training_args_p = TrainingArguments(
    output_dir = "model/participants_classification",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 6,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

trainer_p = Trainer(
    model = model_p,
    args = training_args_p,
    train_dataset = participants_ds["train"],
    eval_dataset = participants_ds["test"],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics_p,
)

In [37]:
trainer_p.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.167940,0.351548,0.226260,0.275321,0.944519
2,0.154898,0.138060,0.337662,0.274326,0.302717,0.947616
3,0.154898,0.151193,0.343284,0.296600,0.318239,0.946852
4,0.103962,0.148526,0.332927,0.320047,0.326360,0.948441
5,0.103962,0.150789,0.349614,0.318875,0.333538,0.949144
6,0.090350,0.144640,0.327374,0.343494,0.335240,0.950653


/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1734, training_loss=0.11223068589295208, metrics={'train_runtime': 325.3194, 'train_samples_per_second': 85.006, 'train_steps_per_second': 5.33, 'total_flos': 3594886826630778.0, 'train_loss': 0.11223068589295208, 'epoch': 6.0})

In [38]:
trainer_p.save_model("model/participants_classification")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Outcomes

In [39]:
doc_ids_o = get_doc_ids("train", "outcomes")
test_doc_ids_o = get_doc_ids("test", "outcomes")

print(f"Number of documents in train split for outcomes: {len(doc_ids_o)}")
print(f"Number of documents in test split for outcomes: {len(test_doc_ids_o)}")

outcomes_tokens = load_documents(doc_ids_o)
test_outcomes_tokens = load_documents(test_doc_ids_o)

Number of documents in train split for outcomes: 4681
Number of documents in test split for outcomes: 190


In [40]:
outcomes_labels = load_labels(doc_ids_o, "outcomes", split="train")
print(f"Length of outcomes_labels: {len(outcomes_labels)}")

test_outcomes_labels = load_labels(test_doc_ids_o, "outcomes", split="test")
print(f"Length of test_outcomes_labels: {len(test_outcomes_labels)}")

outcomes_ner_labels = convert_all_labels_to_ner(outcomes_labels)
test_outcomes_ner_labels = convert_all_labels_to_ner(test_outcomes_labels)

Length of outcomes_labels: 4681
Length of test_outcomes_labels: 190


In [41]:
all_labels_o = chain(*outcomes_ner_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels_o)))

['B-1' 'B-2' 'B-3' 'B-4' 'B-5' 'B-6' 'I-1' 'I-2' 'I-3' 'I-4' 'I-5' 'I-6'
 'O']


In [42]:
id2label_o = {
    0: "O",
    1: "B-1",
    2: "I-1",
    3: "B-2",
    4: "I-2",
    5: "B-3",
    6: "I-3",
    7: "B-4",
    8: "I-4",
    9: "B-5",
    10: "I-5",
    11: "B-6",
    12: "I-6",
}

label2id_o = {
    "O": 0,
    "B-1": 1,
    "I-1": 2,
    "B-2": 3,
    "I-2": 4,
    "B-3": 5,
    "I-3": 6,
    "B-4": 7,
    "I-4": 8,
    "B-5": 9,
    "I-5": 10,
    "B-6": 11,
    "I-6": 12,
}

In [43]:
outcomes_ner_ids = label_to_ids(outcomes_ner_labels, label2id_o)
test_outcomes_ner_ids = label_to_ids(test_outcomes_ner_labels, label2id_o)

In [44]:
outcomes_initial_train_ds = Dataset.from_dict({
    "tokens": outcomes_tokens,
    "ner_tags": outcomes_ner_ids,
})

test_outcomes_initial_train_ds = Dataset.from_dict({
    "tokens": test_outcomes_tokens,
    "ner_tags": test_outcomes_ner_ids,
})

In [45]:
ds_dict_o = DatasetDict({
    "train": outcomes_initial_train_ds,
    "test": test_outcomes_initial_train_ds,
})

In [46]:
outcomes_ds = ds_dict_o.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4681 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

In [47]:
label_list_o = list(label2id_o.keys())

def compute_metrics_o(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_o[p] for (p, l) in zip(prediction, label) if l != -100] # Use this to ignore everything but the first token for words that got split during tokenization
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list_o[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [48]:
model_o = DistilBertForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels = 13, id2label=id2label_o, label2id=label2id_o)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [49]:
training_args_o = TrainingArguments(
    output_dir = "model/outcomes_classification",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 5,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

trainer_o = Trainer(
    model = model_o,
    args = training_args_o,
    train_dataset = outcomes_ds["train"],
    eval_dataset = outcomes_ds["test"],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics_o,
)

In [50]:
trainer_o.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.219902,0.205985,0.210976,0.208451,0.926911
2,0.322726,0.201166,0.247874,0.290805,0.267629,0.930803
3,0.322726,0.188592,0.301307,0.279401,0.289941,0.933597
4,0.223067,0.188002,0.284564,0.302210,0.293121,0.932539
5,0.223067,0.187192,0.288527,0.302922,0.295549,0.933477


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1465, training_loss=0.2502411930227442, metrics={'train_runtime': 275.544, 'train_samples_per_second': 84.941, 'train_steps_per_second': 5.317, 'total_flos': 3042305601861420.0, 'train_loss': 0.2502411930227442, 'epoch': 5.0})

In [51]:
trainer_o.save_model("model/outcomes_classification")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

We can see the same pattern repeated throughout, high accuracy (over 92%) but low on all other scores, indicating that each model is good at predicting negative results it is not good at predicting positive ones

---

## ⚠️ Important: train / val / test issue

In the cells above, `eval_dataset = interventions_ds["test"]` was passed to
the `Trainer`, while `load_best_model_at_end=True`. That means the
early-stopping checkpoint was selected on the *held-out test set* — a form
of dataset leakage that inflates the reported F1.

For the report we should treat the numbers above as upper bounds and
re-train with a proper 90/10 train/val split, evaluating only once on
test. The cell below does this for the `interventions` model. The same
pattern applies to participants and outcomes.

For time, the appended evaluation section below also works with the
*existing* checkpoints — but the report should disclose the leakage
honestly when quoting those numbers.


### Optional: properly re-train interventions with a train/val split


In [52]:
# This is the corrected training setup. Run only if you have time to retrain
# (and to free GPU memory first). Skip if you just want the corrected eval.
RUN_RETRAIN = False   # set to True to actually re-train

if RUN_RETRAIN:
    from sklearn.model_selection import train_test_split
    train_idx, val_idx = train_test_split(
        list(range(len(interventions_ds['train']))),
        test_size=0.1, random_state=42, shuffle=True,
    )
    train_split = interventions_ds['train'].select(train_idx)
    val_split   = interventions_ds['train'].select(val_idx)

    fixed_args = TrainingArguments(
        output_dir='model/interventions_classification_fixed',
        learning_rate=1e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        push_to_hub=False,
        seed=42,
    )

    fixed_trainer = Trainer(
        model=DistilBertForTokenClassification.from_pretrained(
            'distilbert/distilbert-base-uncased',
            num_labels=15, id2label=id2label_i, label2id=label2id_i),
        args=fixed_args,
        train_dataset=train_split,
        eval_dataset=val_split,                        # NOT test
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_i,
    )
    fixed_trainer.train()
    final = fixed_trainer.evaluate(interventions_ds['test'])
    print('Final test metrics (proper val split):', final)
    fixed_trainer.save_model('model/interventions_classification_fixed')


---

## Trustworthy evaluation on the locked eval set

This section evaluates the **already-trained** models from above against
the locked 50-document eval set (same set used by notebooks 02/03/05/06)
so the final comparison is apples-to-apples. We:

1. Load the three saved DistilBERT checkpoints.
2. Run them on the 50 evaluation abstracts.
3. Convert BIO predictions back to surface text spans (one string per field
   per doc).
4. Save predictions and compute coverage / downstream query metrics.


In [53]:
import sys, os
for candidate in ['..', '.', '/content']:
    if os.path.exists(os.path.join(candidate, 'src', 'data.py')):
        if candidate not in sys.path:
            sys.path.insert(0, candidate)
        break

from src.data import (
    ensure_dataset, FIELDS, build_locked_eval_ids, load_eval_doc_ids,
    get_abstract_text, get_gold_spans, load_tokens, load_bio_labels,
)
from src.eval import (
    text_to_token_set, token_overlap_f1, coverage,
    run_queries, save_predictions,
)

ensure_dataset()
EVAL_IDS = load_eval_doc_ids()
if not EVAL_IDS:
    EVAL_IDS = build_locked_eval_ids(n=50)
print(f'Evaluating fine-tuned DistilBERT on {len(EVAL_IDS)} locked test docs.')


Evaluating fine-tuned DistilBERT on 50 locked test docs.


In [54]:
import torch
from transformers import DistilBertForTokenClassification, AutoTokenizer

# Reuse the tokenizer defined earlier in the notebook
eval_tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')

# Try to load the three saved models. If a path is missing the cell skips it.
def try_load(path, num_labels):
    try:
        m = DistilBertForTokenClassification.from_pretrained(path)
        m.eval()
        return m
    except Exception as e:
        print(f'Could not load {path}: {e}')
        return None

models = {
    'participants':  try_load('model/participants_classification', 9),
    'interventions': try_load('model/interventions_classification', 15),
    'outcomes':      try_load('model/outcomes_classification', 13),
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
for m in models.values():
    if m is not None:
        m.to(device)
print('Loaded models:', {k: (v is not None) for k, v in models.items()})


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loaded models: {'participants': True, 'interventions': True, 'outcomes': True}


In [55]:
def predict_spans(model, tokens, max_length=512):
    """Run a token-classification model on a single tokenised document and
    return the surface text spans (joined by ' | ')."""
    if model is None or not tokens:
        return ''

    enc = eval_tokenizer(tokens, is_split_into_words=True,
                         truncation=True, max_length=max_length,
                         return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    pred_ids = torch.argmax(logits, dim=2).squeeze().tolist()
    if not isinstance(pred_ids, list):
        pred_ids = [pred_ids]

    word_ids = enc.word_ids(batch_index=0)
    id2label = model.config.id2label

    # Walk word-aligned predictions and collect contiguous non-O spans
    seen_words = set()
    spans, cur = [], []
    last_label = 'O'
    for wid, pid in zip(word_ids, pred_ids):
        if wid is None or wid in seen_words:
            continue
        seen_words.add(wid)
        if wid >= len(tokens):
            continue
        label = id2label.get(pid, 'O')
        is_entity = (label != 'O')
        if is_entity:
            if not cur:
                cur = [tokens[wid]]
            else:
                cur.append(tokens[wid])
        else:
            if cur:
                spans.append(' '.join(cur))
                cur = []
        last_label = label
    if cur:
        spans.append(' '.join(cur))
    # Dedupe while preserving order
    seen, out = set(), []
    for s in spans:
        sl = s.lower()
        if sl not in seen:
            out.append(s); seen.add(sl)
    return ' | '.join(out)


# Run on every eval doc, every field
predictions_finetuned = {}
for doc_id in EVAL_IDS:
    tokens = load_tokens(doc_id)
    if tokens is None:
        continue
    predictions_finetuned[doc_id] = {
        field: predict_spans(models[field], tokens)
        for field in FIELDS
    }
print(f'Generated predictions for {len(predictions_finetuned)} docs.')
sample = next(iter(predictions_finetuned.items()))
print('\nSample:', sample[0])
for k, v in sample[1].items():
    print(f'  {k:14s}: {v[:120]}')


Generated predictions for 50 docs.

Sample: 10070173
  participants  : seasonal allergic rhinitis | Two hundred and eighty-four
  interventions : budesonide Turbuhaler | budesonide aqua | budesonide Turbuhaler 400 | budesonide aqua 256 | Turbuhaler | budesonide
  outcomes      : quality of life | Mean daily nasal symptom scores | nasal and non-nasal adverse events | epistaxis and headache | nasal 


In [56]:
# Token-overlap F1 + coverage on the eval set
import pandas as pd

rows = []
for field in FIELDS:
    ps, rs, fs = [], [], []
    for d in EVAL_IDS:
        gold = text_to_token_set(' | '.join(get_gold_spans(d, field, split='test')))
        pred = text_to_token_set(predictions_finetuned.get(d, {}).get(field, ''))
        p, r, f = token_overlap_f1(gold, pred)
        ps.append(p); rs.append(r); fs.append(f)
    cov = sum(1 for d in EVAL_IDS if predictions_finetuned.get(d, {}).get(field, '').strip()) / max(1, len(EVAL_IDS))
    rows.append({'approach': 'finetuned_distilbert',
                 'field': field,
                 'precision': sum(ps)/len(ps),
                 'recall':    sum(rs)/len(rs),
                 'f1':        sum(fs)/len(fs),
                 'coverage':  cov})
score_finetuned = pd.DataFrame(rows)
print(score_finetuned.round(3).to_string(index=False))
score_finetuned.to_csv('results_04.csv', index=False)


            approach         field  precision  recall    f1  coverage
finetuned_distilbert  participants      0.855   0.382 0.497      0.92
finetuned_distilbert interventions      0.648   0.842 0.691      0.96
finetuned_distilbert      outcomes      0.804   0.748 0.754      1.00


In [57]:
# Save predictions to disk for the master comparison in notebook 07
save_predictions('finetuned_distilbert', predictions_finetuned)
print('Saved predictions/finetuned_distilbert.json')


Saved predictions/finetuned_distilbert.json


### Downstream usability


In [58]:
# Build gold reference and run the four queries
gold_per_doc = {d: {f: ' | '.join(get_gold_spans(d, f, split='test')) for f in FIELDS} for d in EVAL_IDS}

print(f'{"query":<32s} {"matches":>8s} {"gold":>5s}')
print('-' * 50)
for query_result in run_queries(gold_per_doc):
    desc = query_result['query']
    gold_n = query_result['matches']
    field = query_result['field']
    keyword = query_result['keyword']
    pred_n = sum(1 for d in EVAL_IDS
                 if keyword.lower() in predictions_finetuned.get(d, {}).get(field, '').lower())
    print(f'{desc:<32s} {pred_n:>8d} {gold_n:>5d}')


query                             matches  gold
--------------------------------------------------
Trials with placebo                    12    11
Trials mentioning patients              4    33
Trials measuring pain                   5     5
Trials with rehabilitation              0     0


### Error analysis — worst examples per field


In [59]:
def f1_for_doc(doc_id, field):
    gold = text_to_token_set(' | '.join(get_gold_spans(doc_id, field, split='test')))
    pred = text_to_token_set(predictions_finetuned.get(doc_id, {}).get(field, ''))
    _, _, f = token_overlap_f1(gold, pred)
    return f, gold, pred


for field in FIELDS:
    print('=' * 70)
    print(f'Worst fine-tuned DistilBERT extractions: {field}')
    print('=' * 70)
    items = [(d, *f1_for_doc(d, field)) for d in EVAL_IDS]
    items.sort(key=lambda x: x[1])
    for doc_id, f1, gold, pred in items[:2]:
        print(f'  Doc {doc_id} | F1={f1:.2f}')
        print(f'    Gold: {sorted(list(gold))[:8]}')
        print(f'    Pred: {sorted(list(pred))[:8]}')
        print(f'    Abstract: {get_abstract_text(doc_id)[:200]}')
        print()


Worst fine-tuned DistilBERT extractions: participants
  Doc 10940525 | F1=0.00
    Gold: ['189', '19', '220', '347', '397', '673', 'adult', 'allergy']
    Pred: []
    Abstract: Efficacy and safety of selamectin against fleas and heartworms in dogs and cats presented as veterinary patients in North America . A series of randomized , controlled , masked field studies was condu

  Doc 11381289 | F1=0.00
    Gold: ['at', 'cardiovascular', 'events', 'for', 'high', 'patients', 'risk']
    Pred: []
    Abstract: Why were the results of the Heart Outcomes Prevention Evaluation ( HOPE ) trial so astounding ? The Heart Outcomes Prevention Evaluation ( HOPE ) study was important because it showed the benefits of 

Worst fine-tuned DistilBERT extractions: interventions
  Doc 11750293 | F1=0.00
    Gold: ['clinic', 'outpatient', 'standard', 'treatment']
    Pred: []
    Abstract: The course of depression in recent onset rheumatoid arthritis : the predictive role of disability , illness perceptions

---

## Inference on the locked 50-doc eval set

We now run all three trained DistilBERT models (Participants, Interventions, Outcomes) on the project's locked 50-document test set 

In [60]:
import sys, os
for candidate in ['..', '.', '/content']:
    if os.path.exists(os.path.join(candidate, 'src', 'data.py')):
        if candidate not in sys.path:
            sys.path.insert(0, candidate)
        break

from src.data import load_eval_doc_ids, build_locked_eval_ids, get_abstract_text, FIELDS
from src.eval import save_predictions

EVAL_IDS = load_eval_doc_ids() or build_locked_eval_ids(n=50)
print(f"Running inference on {len(EVAL_IDS)} locked test docs.")

Running inference on 50 locked test docs.


In [61]:
import torch
from transformers import DistilBertForTokenClassification, AutoTokenizer

# Reload tokenizer in case kernel was restarted
infer_tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

# Load all three saved models
MODELS = {
    "participants":  DistilBertForTokenClassification.from_pretrained("model/participants_classification"),
    "interventions": DistilBertForTokenClassification.from_pretrained("model/interventions_classification"),
    "outcomes":      DistilBertForTokenClassification.from_pretrained("model/outcomes_classification"),
}
for m in MODELS.values():
    m.eval()
print("Loaded all three DistilBERT models.")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loaded all three DistilBERT models.


In [62]:
def predict_field_for_doc(doc_id, field, max_len=512):
    """Run the trained DistilBERT for one field on one document.
    Returns the surface text of all predicted entity spans, joined with ' | '.
    """
    tokens = get_abstract_text(doc_id).split()
    if not tokens:
        return ""

    model = MODELS[field]

    enc = infer_tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=max_len,
        padding="max_length",
        return_tensors="pt",
    )

    with torch.no_grad():
        logits = model(**enc).logits
    pred_ids = torch.argmax(logits, dim=2).squeeze().tolist()

    word_ids = enc.word_ids(batch_index=0)
    id2label = model.config.id2label

    # Collect first-subtoken predictions per word
    word_tags = []
    prev_wid = None
    for pid, wid in zip(pred_ids, word_ids):
        if wid is None or wid == prev_wid:
            continue
        word_tags.append((wid, id2label[pid]))
        prev_wid = wid

    # Build spans: any non-O label is part of an entity
    spans, current = [], []
    for wid, tag in word_tags:
        if tag == "O":
            if current:
                spans.append(" ".join(current))
                current = []
        else:
            if wid < len(tokens):
                current.append(tokens[wid])
    if current:
        spans.append(" ".join(current))

    # Deduplicate
    seen, out = set(), []
    for s in spans:
        sl = s.lower()
        if sl and sl not in seen:
            out.append(s)
            seen.add(sl)
    return " | ".join(out)


# Smoke test on first eval doc
sample_id = EVAL_IDS[0]
print(f"Doc {sample_id}:")
for field in FIELDS:
    pred = predict_field_for_doc(sample_id, field)
    print(f"  {field:14s}: {pred[:120]}")

Doc 10070173:
  participants  : seasonal allergic rhinitis | Two hundred and eighty-four
  interventions : budesonide Turbuhaler | budesonide aqua | budesonide Turbuhaler 400 | budesonide aqua 256 | Turbuhaler | budesonide
  outcomes      : quality of life | Mean daily nasal symptom scores | nasal and non-nasal adverse events | epistaxis and headache | nasal 


In [63]:
finetuned_predictions = {}
for i, doc_id in enumerate(EVAL_IDS, 1):
    finetuned_predictions[doc_id] = {
        field: predict_field_for_doc(doc_id, field) for field in FIELDS
    }
    if i % 10 == 0:
        print(f"  [{i}/{len(EVAL_IDS)}] done")

save_predictions("finetuned_distilbert", finetuned_predictions)
print(f"Saved finetuned_distilbert.json with {len(finetuned_predictions)} docs.")

  [10/50] done
  [20/50] done
  [30/50] done
  [40/50] done
  [50/50] done
Saved finetuned_distilbert.json with 50 docs.


In [64]:
from src.eval import load_predictions, text_to_token_set, token_overlap_f1
from src.data import get_gold_spans

reloaded = load_predictions("finetuned_distilbert")
print(f"Loaded {len(reloaded)} doc predictions.")

# Quick F1 spot check on first 5 docs
import numpy as np
for field in FIELDS:
    f1s = []
    for d in EVAL_IDS[:10]:
        gold = text_to_token_set(' | '.join(get_gold_spans(d, field, split='test')))
        pred = text_to_token_set(reloaded.get(d, {}).get(field, ''))
        _, _, f = token_overlap_f1(gold, pred)
        f1s.append(f)
    print(f"  {field:14s}  10-doc F1: {np.mean(f1s):.3f}")

Loaded 50 doc predictions.
  participants    10-doc F1: 0.444
  interventions   10-doc F1: 0.579
  outcomes        10-doc F1: 0.748
